# Early Fusion with CodeT5+ Encoder Only
## Efficient Multimodal Classification

In [4]:
import os
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, T5EncoderModel, AutoImageProcessor, ViTModel
from PIL import Image
from tqdm import tqdm
import numpy as np

In [5]:
# Configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = "../checkpoints/early_fusion_ct5p_encoder/"
BATCH_SIZE = 8
EPOCHS = 3
LR = 2e-5

CT5P_CKPT = "../checkpoints/ct5p_only/checkpoint-2820/"
VIT_CKPT = "../checkpoints/vit_only/deit_epoch_3.pt"

TEXT_DIR = "../text_files/train"
IMAGE_DIR = "../snapshots/train"
TEST_BASE = "../snapshots"

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [6]:
class CodeT5ForSequenceClassification(nn.Module):
    """
    Custom CodeT5+ model for sequence classification using only the encoder
    """
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.encoder = T5EncoderModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.encoder.config.d_model, num_labels)
        self.num_labels = num_labels
        
    def forward(self, input_ids, attention_mask=None, labels=None):
        # Get encoder outputs
        outputs = self.encoder(
            input_ids=input_ids, 
            attention_mask=attention_mask,
            return_dict=True
        )
        
        # Use mean pooling over sequence dimension
        sequence_output = outputs.last_hidden_state
        
        # Apply attention mask for mean pooling
        if attention_mask is not None:
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(sequence_output.size()).float()
            sum_embeddings = torch.sum(sequence_output * input_mask_expanded, 1)
            sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
            pooled_output = sum_embeddings / sum_mask
        else:
            pooled_output = sequence_output.mean(dim=1)
        
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
        
        return {'loss': loss, 'logits': logits}

In [7]:
# Load CodeT5+ Encoder Only
print("Loading CodeT5+ Encoder...")
tokenizer = AutoTokenizer.from_pretrained("Salesforce/codet5p-220m")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

from transformers import T5EncoderModel

# 1️⃣ Load your fine-tuned classification model
model = CodeT5ForSequenceClassification("Salesforce/codet5p-220m", 2)
checkpoint = torch.load(os.path.join(CT5P_CKPT, "pytorch_model.bin"), map_location=DEVICE)
model.load_state_dict(checkpoint, strict=False)
model.to(DEVICE)
model.eval()

# print("✓ Full fine-tuned CodeT5+ classification model loaded.")

# 2️⃣ Extract the encoder weights only
encoder_state_dict = model.encoder.state_dict()

# 3️⃣ Load them into a fresh encoder model
text_model = T5EncoderModel.from_pretrained("Salesforce/codet5p-220m")
text_model.load_state_dict(encoder_state_dict, strict=False)
text_model.to(DEVICE)
text_model.eval()

# print("✓ Encoder extracted successfully from fine-tuned model.")

Loading CodeT5+ Encoder...


T5EncoderModel(
  (shared): Embedding(32100, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32100, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=768, out_features=3072, bias=False)
              (wo): Linear(in_features=3072, out_features=768, bias=False)
              (dropout): Dropout(p=0.1, 

In [5]:
# Load ViT model
print("Loading ViT model...")
image_processor = AutoImageProcessor.from_pretrained("facebook/deit-base-patch16-224")

# Load your trained classification model first
from transformers import ViTForImageClassification
trained_model = ViTForImageClassification.from_pretrained(
    "facebook/deit-base-patch16-224",
    num_labels=2,
    ignore_mismatched_sizes=True
)
trained_model.load_state_dict(torch.load(VIT_CKPT, map_location=DEVICE))
trained_model.to(DEVICE)

# Extract just the ViT base model for embeddings
vit_model = trained_model.vit
vit_model.to(DEVICE)
vit_model.eval()
print("✓ ViT model loaded successfully")

Loading ViT model...


Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.
Some weights of ViTForImageClassification were not initialized from the model checkpoint at facebook/deit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([2]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ ViT model loaded successfully


In [6]:
class FusionDataset(Dataset):
    def __init__(self, text_dir, image_dir, tokenizer, image_processor):
        self.text_paths = []
        self.image_paths = []
        self.labels = []
        self.tokenizer = tokenizer
        self.image_processor = image_processor

        # Scan text folders
        for label_folder in sorted(os.listdir(text_dir)):
            label_path = os.path.join(text_dir, label_folder)
            if not os.path.isdir(label_path):
                continue
            label = int(label_folder.split("_")[1])  # e.g., "Label_0" -> 0
            for txt_file in sorted(os.listdir(label_path)):
                if txt_file.endswith(".txt"):
                    self.text_paths.append(os.path.join(label_path, txt_file))
                    self.labels.append(label)

        # Scan image folders
        self.image_paths = []
        for label_folder in sorted(os.listdir(image_dir)):
            label_path = os.path.join(image_dir, label_folder)
            if not os.path.isdir(label_path):
                continue
            for img_file in sorted(os.listdir(label_path)):
                if img_file.lower().endswith((".png", ".jpg", ".jpeg")):
                    self.image_paths.append(os.path.join(label_path, img_file))

        # Ensure text_paths and image_paths are aligned
        assert len(self.text_paths) == len(self.image_paths), "Text and image counts must match!"
        print(f"✓ Loaded {len(self.text_paths)} samples")

    def __len__(self):
        return len(self.text_paths)

    def __getitem__(self, idx):
        # ----- TEXT -----
        with open(self.text_paths[idx], "r", encoding='utf-8', errors='ignore') as f:
            text = f.read()
        encoding = self.tokenizer(
            text, 
            return_tensors="pt", 
            truncation=True, 
            padding="max_length", 
            max_length=512
        )
        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)

        # ----- IMAGE -----
        image = Image.open(self.image_paths[idx]).convert("RGB")
        image_tensor = self.image_processor(images=image, return_tensors="pt")
        for k in image_tensor:
            image_tensor[k] = image_tensor[k].squeeze(0)

        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return input_ids, attention_mask, image_tensor, label

In [7]:
class FusionClassifier(nn.Module):
    def __init__(self, text_model, vit_model, hidden_dim=512, num_classes=2):
        super().__init__()
        self.text_model = text_model
        self.vit_model = vit_model

        # Freeze backbone models
        for p in self.text_model.parameters():
            p.requires_grad = False
        for p in self.vit_model.parameters():
            p.requires_grad = False

        # Get embedding dimensions
        text_emb_dim = text_model.config.d_model  # T5 uses d_model instead of hidden_size
        vit_emb_dim = vit_model.config.hidden_size

        self.classifier = nn.Sequential(
            nn.Linear(text_emb_dim + vit_emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, input_ids, attention_mask, image_tensor):
        # Text embedding: Mean pooling of encoder outputs
        text_outputs = self.text_model(input_ids=input_ids, attention_mask=attention_mask)
        
        # For T5 encoder, we use mean pooling over the sequence
        text_embeddings = text_outputs.last_hidden_state  # [batch, seq_len, hidden]
        
        # Apply attention mask for mean pooling
        if attention_mask is not None:
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(text_embeddings.size()).float()
            sum_embeddings = torch.sum(text_embeddings * input_mask_expanded, 1)
            sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
            text_cls = sum_embeddings / sum_mask
        else:
            text_cls = text_embeddings.mean(dim=1)

        # Image embedding: CLS token
        image_outputs = self.vit_model(**{k: v for k, v in image_tensor.items()})
        image_cls = image_outputs.last_hidden_state[:, 0, :]  # [batch, hidden]

        # Concatenate and classify
        fused = torch.cat([text_cls, image_cls], dim=1)
        logits = self.classifier(fused)
        return logits

In [8]:
# Load dataset
print("Loading training dataset...")
dataset = FusionDataset(TEXT_DIR, IMAGE_DIR, tokenizer, image_processor)
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# Initialize model
model = FusionClassifier(text_model, vit_model).to(DEVICE)
optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

print(f"✓ Model initialized on {DEVICE}")
print(f"✓ Training samples: {len(dataset)}")
print(f"✓ Batch size: {BATCH_SIZE}")
print(f"✓ Total parameters: {sum(p.numel() for p in model.classifier.parameters()):,}")

Loading training dataset...
✓ Loaded 7520 samples
✓ Model initialized on cuda
✓ Training samples: 7520
✓ Batch size: 8
✓ Total parameters: 787,970


In [9]:
print("Starting training...")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for batch in progress_bar:
        input_ids, attention_mask, image_tensor, labels = batch
        
        # Move to device
        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        labels = labels.to(DEVICE)
        for k in image_tensor:
            image_tensor[k] = image_tensor[k].to(DEVICE)

        # Forward pass
        logits = model(input_ids, attention_mask, image_tensor)
        loss = criterion(logits, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})
    
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} | Avg Loss: {avg_loss:.4f}")

    # Save checkpoint
    ckpt_path = os.path.join(OUTPUT_DIR, f"fusion_ct5p_encoder_epoch_{epoch+1}.pt")
    torch.save(model.state_dict(), ckpt_path)
    print(f"✓ Checkpoint saved: {ckpt_path}")

print("✓ Training completed!")

Starting training...


Epoch 1/3: 100%|██████████| 940/940 [01:50<00:00,  8.51it/s, loss=0.1315]


Epoch 1/3 | Avg Loss: 0.1935
✓ Checkpoint saved: ../checkpoints/early_fusion_ct5p_encoder/fusion_ct5p_encoder_epoch_1.pt


Epoch 2/3: 100%|██████████| 940/940 [01:50<00:00,  8.48it/s, loss=0.0297]


Epoch 2/3 | Avg Loss: 0.1676
✓ Checkpoint saved: ../checkpoints/early_fusion_ct5p_encoder/fusion_ct5p_encoder_epoch_2.pt


Epoch 3/3: 100%|██████████| 940/940 [01:50<00:00,  8.48it/s, loss=0.0280]


Epoch 3/3 | Avg Loss: 0.1683
✓ Checkpoint saved: ../checkpoints/early_fusion_ct5p_encoder/fusion_ct5p_encoder_epoch_3.pt
✓ Training completed!


# Evaluation

In [11]:
def test_on_dataset(text_dir, image_dir, tokenizer, image_processor, fusion_model, batch_size=4):
    """Test the fusion model on a single dataset"""

    dataset = FusionDataset(text_dir, image_dir, tokenizer, image_processor)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    fusion_model.eval()
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc="Testing"):
            input_ids, attention_mask, image_tensor, labels = batch

            # Move to device
            input_ids = input_ids.to(DEVICE)
            attention_mask = attention_mask.to(DEVICE)
            labels = labels.to(DEVICE)
            for k in image_tensor:
                image_tensor[k] = image_tensor[k].to(DEVICE)

            # Forward pass
            logits = fusion_model(input_ids, attention_mask, image_tensor)
            preds = torch.argmax(logits, dim=1)

            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)

    accuracy = total_correct / total_samples
    print(f"✓ Dataset Accuracy: {accuracy*100:.2f}%")
    return accuracy

In [13]:
# Load the best checkpoint for evaluation
FUSION_CKPT = os.path.join(OUTPUT_DIR, "fusion_ct5p_encoder_epoch_3.pt")

# Reinitialize model
eval_model = FusionClassifier(text_model, vit_model)
eval_model.to(DEVICE)

# Load checkpoint
state_dict = torch.load(FUSION_CKPT, map_location=DEVICE)
eval_model.load_state_dict(state_dict)
eval_model.eval()
print("✓ Evaluation model loaded successfully")

✓ Evaluation model loaded successfully


In [14]:
# Test on all test datasets
print("Running evaluation on test datasets...")
accuracies = {}

test_name = f"valid"
print(f"\n{test_name}:")

accuracy = test_on_dataset(
    f"../text_files/{test_name}", 
    f"../snapshots/{test_name}", 
    tokenizer, 
    image_processor, 
    eval_model, 
    batch_size=4
)
accuracies[test_name] = accuracy

# Print summary
print("\n" + "="*50)
print("SUMMARY OF RESULTS")
print("="*50)
for test_name, acc in accuracies.items():
    print(f"{test_name}: {acc*100:.2f}%")

avg_accuracy = np.mean(list(accuracies.values()))
print(f"\nAverage Accuracy: {avg_accuracy*100:.2f}%")
print("✓ Evaluation completed!")

Running evaluation on test datasets...

valid:
✓ Loaded 1880 samples


Testing: 100%|██████████| 470/470 [00:24<00:00, 19.19it/s]

✓ Dataset Accuracy: 86.81%

SUMMARY OF RESULTS
valid: 86.81%

Average Accuracy: 86.81%
✓ Evaluation completed!


In [ ]:
#Epoch 3 : 87.07 % = Validation Accuracy
#Epoch 2 : 87.18 % = Validation Accuracy
#Epoch 1 : 86.18 % = Validation Accuracy